# Real Estate Property Price Predictor

Week 7 Milestone — dual-regressor training, metric benchmarking, and custom property inference.

In [ ]:
import math
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("data/property_data.csv")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

forest_model = RandomForestRegressor(
    n_estimators=250,
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1
)

linear_model.fit(X_train, y_train)
forest_model.fit(X_train, y_train)

linear_pred = linear_model.predict(X_test)
forest_pred = forest_model.predict(X_test)

In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": math.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame([
    {"Model": "LinearRegression", **regression_metrics(y_test, linear_pred)},
    {"Model": "RandomForestRegressor", **regression_metrics(y_test, forest_pred)}
])

results

## Custom Property Inference

In [ ]:
custom_property = pd.DataFrame([{
    "bedrooms": 3,
    "bathrooms": 2.0,
    "square_feet": 1200,
    "house_age": 10,
    "lot_size": 2500,
    "population": 450,
    "median_income": 6.5,
    "latitude": 37.77,
    "longitude": -122.42
}])

print("Linear Regression:", f"${linear_model.predict(custom_property)[0]:,.0f}")
print("Random Forest:", f"${forest_model.predict(custom_property)[0]:,.0f}")

## Conclusion

Both models are trained on the same 80/20 split. MAE, MSE, RMSE and R² are used for benchmarking. The custom feature vector demonstrates how a new property can be passed to the trained regressors.